# Selective Activation Checkpointing — Trading Memory for Compute

**What you'll learn:**
- Why activations dominate GPU memory during training
- Basic activation checkpointing with `torch.utils.checkpoint`
- Selective checkpointing with per-op policies
- Integration with `torch.compile`
- When and where to apply checkpointing

**Prerequisites:** Understanding of autograd, backpropagation, and training large models

In [ ]:
import torch
import torch.nn as nn
import torch.utils.checkpoint as checkpoint_utils
import tracemalloc
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## The Memory Problem: Activations Dominate

During forward pass, PyTorch saves **intermediate activations** (outputs of each layer) because they're needed for backpropagation. For deep models, this memory cost dwarfs the model parameters themselves.

**Example:** A Transformer with 12 layers, batch size 32, sequence length 512:
- Parameters: ~110M × 4 bytes ≈ 440 MB
- Activations: Can easily reach **2-4 GB** or more

This is the fundamental tension: you need activations for backward, but they consume enormous memory.

In [ ]:
# Visualize activation memory growth in a deep network
class DeepMLP(nn.Module):
    def __init__(self, width=512, depth=10):
        super().__init__()
        self.layers = nn.ModuleList(
            [nn.Linear(width, width) for _ in range(depth)]
        )

    def forward(self, x):
        for layer in self.layers:
            x = torch.relu(layer(x))
        return x

model = DeepMLP(width=512, depth=10)
x = torch.randn(64, 512)

# Count parameters vs activation memory
param_memory = sum(p.numel() * p.element_size() for p in model.parameters())
print(f"Parameter memory: {param_memory / 1024:.1f} KB")

# Each layer saves its output for backward — 10 layers × (64 × 512 × 4 bytes)
activation_per_layer = 64 * 512 * 4  # batch × width × float32
total_activation = activation_per_layer * 10
print(f"Activation memory (approx): {total_activation / 1024:.1f} KB")
print(f"Activation / Parameter ratio: {total_activation / param_memory:.1f}x")

## Basic Activation Checkpointing

The idea is simple: **don't save activations during forward pass**. Instead, recompute them during backward. This trades compute time (~33% more) for massive memory savings.

`torch.utils.checkpoint.checkpoint` wraps a function so that its intermediate activations are discarded and recomputed during backward.

In [ ]:
# Compare memory: with and without checkpointing
class TransformerBlock(nn.Module):
    def __init__(self, d_model=256, nhead=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm1(x + self.attn(x, x, x)[0])
        x = self.norm2(x + self.ff(x))
        return x

class SimpleTransformer(nn.Module):
    def __init__(self, num_layers=6, d_model=256, nhead=4):
        super().__init__()
        self.layers = nn.ModuleList(
            [TransformerBlock(d_model, nhead) for _ in range(num_layers)]
        )

    def forward(self, x, use_checkpoint=False):
        for layer in self.layers:
            if use_checkpoint:
                x = checkpoint_utils.checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return x

model = SimpleTransformer(num_layers=6, d_model=256)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Transformer blocks: {len(model.layers)}")

### Memory Measurement

Let's measure actual memory usage. We'll track tensor allocations before and after forward+backward:

In [ ]:
import gc

def measure_memory(model, x, use_checkpoint=False):
    """Measure peak memory during forward + backward on CPU using tracemalloc."""
    gc.collect()
    tracemalloc.start()

    # Forward
    out = model(x, use_checkpoint=use_checkpoint)
    loss = out.sum()

    _, peak_fwd = tracemalloc.get_traced_memory()

    # Backward
    loss.backward()

    _, peak_total = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return peak_fwd, peak_total

x = torch.randn(32, 64, 256, requires_grad=True)

# Without checkpointing
peak_fwd_no_ckpt, peak_total_no_ckpt = measure_memory(model, x, use_checkpoint=False)

# With checkpointing
x2 = x.detach().requires_grad_(True)
peak_fwd_ckpt, peak_total_ckpt = measure_memory(model, x2, use_checkpoint=True)

print("Memory usage (CPU tracemalloc, approximate):")
print(f"  Without checkpoint — Forward peak: {peak_fwd_no_ckpt/1024/1024:.1f} MB, Total peak: {peak_total_no_ckpt/1024/1024:.1f} MB")
print(f"  With checkpoint    — Forward peak: {peak_fwd_ckpt/1024/1024:.1f} MB, Total peak: {peak_total_ckpt/1024/1024:.1f} MB")
print(f"\nMemory reduction: {(1 - peak_total_ckpt/peak_total_no_ckpt)*100:.0f}%")

## How Checkpointing Works Under the Hood

During **forward pass**: the checkpointed function runs normally but intermediate tensors (within the function) are **not saved** for backward. Only the inputs and outputs are kept.

During **backward pass**: when gradients reach the checkpoint boundary, the forward pass is **re-executed** to recreate the intermediates, then backward proceeds normally.

```
Without checkpoint:     With checkpoint:
Forward:                Forward:
  Save act1              (discard act1)
  Save act2              (discard act2)
  Save act3              (discard act3)

Backward:               Backward:
  Use act3 → grad3       Recompute act1,2,3
  Use act2 → grad2       Use act3 → grad3
  Use act1 → grad1       Use act2 → grad2
                          Use act1 → grad1
```

In [ ]:
# Demonstrate that checkpointing produces correct gradients
model_a = SimpleTransformer(num_layers=4, d_model=64)
model_b = SimpleTransformer(num_layers=4, d_model=64)
model_b.load_state_dict(model_a.state_dict())

x = torch.randn(4, 16, 64)

# Forward + backward without checkpoint
out_a = model_a(x, use_checkpoint=False)
out_a.sum().backward()

# Forward + backward with checkpoint
out_b = model_b(x, use_checkpoint=True)
out_b.sum().backward()

# Verify outputs and gradients match
print(f"Outputs match: {torch.allclose(out_a, out_b, atol=1e-6)}")
for (name_a, p_a), (name_b, p_b) in zip(
    model_a.named_parameters(), model_b.named_parameters()
):
    if p_a.grad is not None:
        match = torch.allclose(p_a.grad, p_b.grad, atol=1e-5)
        if not match:
            print(f"  {name_a}: gradients MISMATCH")
            break
else:
    print("All gradients match! Checkpointing is mathematically equivalent.")

## Selective Activation Checkpointing

Basic checkpointing is all-or-nothing for each function: either save everything or recompute everything. **Selective checkpointing** gives finer control — you can decide per-operation what to save vs recompute.

The key insight: not all operations are equal.
- **Expensive ops** (matmul, attention): save these, don't recompute
- **Cheap ops** (relu, add, dropout): recompute these, don't waste memory saving them

In [ ]:
from torch.utils.checkpoint import (
    CheckpointPolicy,
    create_selective_checkpoint_contexts,
)

# Define a policy: save expensive ops, recompute cheap ones
def selective_policy(ctx, func, *args, **kwargs):
    """Save matmul/attention results, recompute activations like relu."""
    op_name = str(func)
    # These are expensive to recompute — save them
    if any(expensive in op_name for expensive in ['mm', 'bmm', 'addmm', 'scaled_dot']):
        return CheckpointPolicy.MUST_SAVE
    # These are cheap to recompute — don't save them
    return CheckpointPolicy.PREFER_RECOMPUTE

print("Selective checkpoint policy defined:")
print("  MUST_SAVE: mm, bmm, addmm (matrix multiplies)")
print("  PREFER_RECOMPUTE: everything else (relu, add, layernorm, etc.)")
print("\nThis gives most of the memory savings while avoiding")
print("recomputing the most expensive operations.")

### Using create_selective_checkpoint_contexts

The `create_selective_checkpoint_contexts` function creates the context managers needed by `torch.utils.checkpoint.checkpoint`:

In [ ]:
# Using selective checkpointing with a custom model
class SelectiveTransformer(nn.Module):
    def __init__(self, num_layers=6, d_model=128, nhead=4):
        super().__init__()
        self.layers = nn.ModuleList(
            [TransformerBlock(d_model, nhead) for _ in range(num_layers)]
        )

    def forward(self, x, use_selective_checkpoint=False):
        for layer in self.layers:
            if use_selective_checkpoint:
                context_fn = lambda: create_selective_checkpoint_contexts(selective_policy)
                x = checkpoint_utils.checkpoint(
                    layer, x, use_reentrant=False, context_fn=context_fn
                )
            else:
                x = layer(x)
        return x

sel_model = SelectiveTransformer(num_layers=4, d_model=128)
x = torch.randn(8, 32, 128, requires_grad=True)

# Run with selective checkpointing
out = sel_model(x, use_selective_checkpoint=True)
out.sum().backward()

print(f"Selective checkpoint forward+backward completed")
print(f"Output shape: {out.shape}")
print(f"Input grad shape: {x.grad.shape}")
print(f"Input grad norm: {x.grad.norm():.4f}")

## Integration with torch.compile

Selective checkpointing works seamlessly with `torch.compile`. The compiler can see through the checkpoint boundaries and optimize the overall computation.

In [ ]:
# torch.compile + selective checkpointing
# On CPU, torch.compile uses the inductor backend

class CompiledCheckpointModel(nn.Module):
    def __init__(self, d_model=64, nhead=2, num_layers=4):
        super().__init__()
        self.layers = nn.ModuleList(
            [TransformerBlock(d_model, nhead) for _ in range(num_layers)]
        )

    def forward(self, x):
        for layer in self.layers:
            context_fn = lambda: create_selective_checkpoint_contexts(selective_policy)
            x = checkpoint_utils.checkpoint(
                layer, x, use_reentrant=False, context_fn=context_fn
            )
        return x

compiled_model = CompiledCheckpointModel()

# torch.compile wraps the whole thing — it can optimize across checkpoint boundaries
try:
    compiled_fn = torch.compile(compiled_model, backend="eager")
    x = torch.randn(4, 16, 64, requires_grad=True)
    out = compiled_fn(x)
    out.sum().backward()
    print(f"torch.compile + selective checkpoint: SUCCESS")
    print(f"Output shape: {out.shape}")
    print(f"Grad computed: {x.grad is not None}")
except Exception as e:
    print(f"torch.compile not available in this environment: {e}")
    print("(This works on GPU with inductor backend)")

## When to Use Checkpointing

**Good candidates:**
- Transformer layers (high activation memory, moderate recompute cost)
- Deep convolutional networks (many intermediate feature maps)
- Any model where you're OOM but not compute-bound

**Bad candidates:**
- Already compute-bound models (checkpointing adds ~33% compute)
- Shallow models (minimal activation memory)
- Inference only (no backward pass, no activations saved)

### Granularity Guidelines

| Strategy | Memory Savings | Compute Cost | Best For |
|----------|---------------|--------------|----------|
| No checkpoint | 0% | Baseline | Small models |
| Per-layer checkpoint | ~60-80% | ~33% | Most training |
| Selective checkpoint | ~40-60% | ~10-20% | Large models |
| Every-op checkpoint | ~90%+ | ~100%+ | Extreme cases |

In [ ]:
# checkpoint_sequential: convenience for nn.Sequential models
from torch.utils.checkpoint import checkpoint_sequential

class SequentialModel(nn.Module):
    def __init__(self, width=256, depth=12):
        super().__init__()
        layers = []
        for _ in range(depth):
            layers.extend([nn.Linear(width, width), nn.ReLU()])
        self.layers = nn.Sequential(*layers)
        self.depth = depth

    def forward(self, x, segments=1):
        if segments > 1:
            return checkpoint_sequential(self.layers, segments, x, use_reentrant=False)
        return self.layers(x)

seq_model = SequentialModel(width=128, depth=8)

# Split the 16 layers (8 Linear + 8 ReLU) into 4 checkpoint segments
x = torch.randn(16, 128, requires_grad=True)
out = seq_model(x, segments=4)
out.sum().backward()

print(f"checkpoint_sequential with 4 segments:")
print(f"  Input: {x.shape}")
print(f"  Output: {out.shape}")
print(f"  Gradient computed: {x.grad is not None}")
print(f"  Each segment recomputes ~{len(seq_model.layers)//4} layers during backward")

## Practical Example: Checkpointing a Real Training Loop

Let's put it all together in a training loop that uses checkpointing to handle a model that would otherwise OOM:

In [ ]:
# Full training loop with checkpointing
class CheckpointedTransformer(nn.Module):
    def __init__(self, vocab_size=1000, d_model=128, nhead=4, num_layers=6):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList(
            [TransformerBlock(d_model, nhead) for _ in range(num_layers)]
        )
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, tokens):
        x = self.embedding(tokens)
        for layer in self.layers:
            x = checkpoint_utils.checkpoint(layer, x, use_reentrant=False)
        return self.head(x)

# Training loop
model = CheckpointedTransformer(vocab_size=1000, d_model=128, num_layers=6)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Training with activation checkpointing on all 6 layers\n")

for step in range(5):
    tokens = torch.randint(0, 1000, (16, 64))  # batch=16, seq=64
    logits = model(tokens)
    loss = nn.functional.cross_entropy(
        logits.view(-1, 1000),
        tokens.view(-1),  # simple language modeling
    )
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"  Step {step+1}: loss = {loss.item():.4f}")

print("\nTraining with checkpointing completed successfully!")

In [ ]:
# Demonstrate the memory/compute tradeoff with different segment counts
seq_model2 = SequentialModel(width=128, depth=12)
x = torch.randn(32, 128)

import time

for segments in [1, 2, 4, 6, 12]:
    # Warm up
    out = seq_model2(x.requires_grad_(True), segments=segments)
    out.sum().backward()

    # Time it
    times = []
    for _ in range(10):
        x_t = torch.randn(32, 128, requires_grad=True)
        start = time.perf_counter()
        out = seq_model2(x_t, segments=segments)
        out.sum().backward()
        times.append(time.perf_counter() - start)

    avg_ms = sum(times) / len(times) * 1000
    label = "no checkpoint" if segments == 1 else f"{segments} segments"
    print(f"  {label:20s}: {avg_ms:.2f} ms avg")

### use_reentrant=False vs True

Always prefer `use_reentrant=False` (the new default). The legacy `use_reentrant=True` has subtle bugs:
- Doesn't work with `torch.autograd.graph.saved_tensors_hooks`
- Can silently produce incorrect gradients with certain models
- Doesn't support kwargs in the checkpointed function

```python
# Good
checkpoint_utils.checkpoint(fn, x, use_reentrant=False)

# Legacy (avoid)
checkpoint_utils.checkpoint(fn, x, use_reentrant=True)
```

## 🏋️ Try It Yourself: Add Checkpointing and Measure Memory

**Exercise:** Create a deep convolutional network, add checkpointing, and compare memory usage.

Your task:
1. Create a model with 8+ Conv2d layers
2. Run forward + backward without checkpointing
3. Add checkpointing to every 2 layers
4. Compare memory usage

In [ ]:
# Exercise starter code

class DeepConvNet(nn.Module):
    def __init__(self, num_layers=8, channels=64):
        super().__init__()
        layers = [nn.Conv2d(3, channels, 3, padding=1), nn.ReLU()]
        for _ in range(num_layers - 1):
            layers.extend([nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU()])
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(channels, 10)

    def forward(self, x):
        # TODO: Add checkpointing here
        # Hint: Split self.features into groups of 2 layers
        # and wrap each group with checkpoint_utils.checkpoint
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)

conv_model = DeepConvNet(num_layers=8, channels=64)
x = torch.randn(8, 3, 32, 32, requires_grad=True)

# Your code here:
# 1. Run forward + backward without checkpointing
# 2. Modify forward() to use checkpointing
# 3. Compare memory usage

# Example solution approach:
out = conv_model(x)
out.sum().backward()
print(f"Forward + backward completed")
print(f"Output shape: {out.shape}")
print(f"Model parameters: {sum(p.numel() for p in conv_model.parameters()):,}")
print(f"\nHint: Use checkpoint_sequential(self.features, segments=4, x)")

## Key Takeaways

1. **Activation memory dominates** — for deep models, activations during forward pass consume far more memory than model parameters

2. **Basic checkpointing** (`torch.utils.checkpoint.checkpoint`) discards activations during forward and recomputes them during backward, saving ~60-80% memory at ~33% compute cost

3. **Selective checkpointing** (`create_selective_checkpoint_contexts`) gives per-op control — save expensive operations (matmul), recompute cheap ones (relu, add)

4. **Correctness is preserved** — checkpointing produces mathematically identical gradients (verified by comparing with non-checkpointed training)

5. **Works with torch.compile** — the compiler can optimize across checkpoint boundaries

6. **checkpoint_sequential** provides a convenient API for `nn.Sequential` models

7. **Use `use_reentrant=False`** — the non-reentrant version is the recommended default; it handles more cases correctly

8. **Measure before optimizing** — always measure actual memory savings to verify checkpointing helps your specific model